<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Phase_1_Framework_Generation/FRAMEWORK_GENERATION_PHASE_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 📦 CELL 1: Unified Dependency Installation
# ============================================================

# We combine everything into one command so pip can find a compatible version of openai.
# 'crewai[tools]' already includes crewai, so we don't need to list both.
!pip install -U "crewai[tools]" litellm anthropic duckduckgo-search openpyxl tavily-python -q

# IMPORTANT: Force a specific OpenAI version that balances the two if conflicts persist
!pip install "openai>=1.83.0,<2.0.0" -q

In [ ]:
# ============================================================
# 🔑 CELL 2 UPDATED: API Keys + Model Config
# ============================================================

import os
from openai import OpenAI
from anthropic import Anthropic
from tavily import TavilyClient
from crewai import Agent, Task, Crew
from crewai.tools import tool

os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["DEEPSEEK_API_KEY"] = ""


# ── Model config ──────────────────────────────────────────────
ASSESSOR_1_MODEL   = "deepseek-chat"
ASSESSOR_2_MODEL   = "gpt-4o"
CONSOLIDATOR_MODEL = "claude-opus-4-6"
EXTRACTOR_MODEL    = "claude-sonnet-4-6"

ASSESSOR_1_LABEL   = "DeepSeek-Assessor"
ASSESSOR_2_LABEL   = "GPT4o-Assessor"
CONSOLIDATOR_LABEL = "Claude-Consolidator"
EXTRACTOR_LABEL    = "Claude-Extractor"

print(f"✅ Configuration Loaded:")
print(f" 📌 Assessor 1: {ASSESSOR_1_MODEL}")
print(f" 📌 Assessor 2: {ASSESSOR_2_MODEL}")

# ── Test OpenAI (New) ──────────────────────────────────────────
print("\n--- 🧪 Testing OpenAI (Assessor 2) ---")
try:
    # Uses default base_url for OpenAI
    client_openai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    r_oa = client_openai.chat.completions.create(
        model=ASSESSOR_2_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ OpenAI works! → {r_oa.choices[0].message.content}")
except Exception as e:
    print(f"❌ OpenAI failed: {e}")

# ── Test DeepSeek ─────────────────────────────────────────────
print("\n--- 🧪 Testing DeepSeek (Assessor 1) ---")
try:
    # Explicitly set DeepSeek base_url
    client_deepseek = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"],
        base_url="https://api.deepseek.com"
    )
    r_ds = client_deepseek.chat.completions.create(
        model=ASSESSOR_1_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ DeepSeek works! → {r_ds.choices[0].message.content}")
except Exception as e:
    print(f"❌ DeepSeek failed: {e}")

# ── Test Claude ───────────────────────────────────────────────
print("\n--- 🧪 Testing Claude (Consolidator) ---")
try:
    client_claude = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    r_c = client_claude.messages.create(
        model=CONSOLIDATOR_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ Claude works! → {r_c.content[0].text}")
except Exception as e:
    print(f"❌ Claude failed: {e}")

print("\n" + "=" * 50)
print("🚀 All APIs tested and routing confirmed!")
print("=" * 50)

In [ ]:
# ============================================================
# T2D literature search with full page content
# ============================================================

# ── 2. Custom Tavily Tool ─────────────────────────────────────
@tool("Deep Medical Literature Search")
def medical_literature_search(query: str) -> str:
    """
    Searches the web for clinical guidelines, textbooks, and medical literature.
    Returns the full, raw text content of the top results for deep analysis.
    Use this when you need detailed clinical criteria, grading scales, or tables.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    response = client.search(
        query=query,
        max_results=3,
        search_depth="advanced",
        include_raw_content=True
    )

    results_str = ""
    for result in response.get("results", []):
        results_str += f"Title: {result['title']}\nURL: {result['url']}\n"
        content = result.get('raw_content', result['content'])
        results_str += f"Content:\n{content[:15000]}\n\n---\n\n"

    return results_str

# ── 3. Define the Agents (Using LiteLLM string format) ────────
print("\n🔵 Initializing Parallel Agents...")

# Assessor 1: GPT-4o
researcher_gpt = Agent(
    role='Senior Clinical Researcher (OpenAI)',
    goal='Find precise clinical criteria for classifying T2D severity.',
    backstory='You are a methodical medical researcher specializing in EHR phenotyping.',
    tools=[medical_literature_search],
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# Assessor 2: DeepSeek
# Note: "deepseek/" prefix tells litellm to route to the DeepSeek API
researcher_deepseek = Agent(
    role='Senior Clinical Researcher (DeepSeek)',
    goal='Find precise clinical criteria for classifying T2D severity.',
    backstory='You are a methodical medical researcher specializing in EHR phenotyping.',
    tools=[medical_literature_search],
    verbose=True,
    allow_delegation=False,
    llm="deepseek/deepseek-chat"
)

# Consolidator: Claude 3.5 Sonnet
# Note: "anthropic/" prefix tells litellm to route to the Anthropic API
consolidator_claude = Agent(
    role='Chief Medical Informatician',
    goal='Synthesize multiple research reports into a single, definitive, operational EHR phenotyping framework.',
    backstory='You are a senior physician and informatician. You excel at taking different proposed clinical criteria, resolving discrepancies, and formatting them into strict rules.',
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

# ── 4. Define the Tasks ───────────────────────────────────────
print("🔵 Defining Tasks...")

task_description = 'Search for T2D severity phenotyping criteria considering comorbidities and natural history. Return a bulleted list of the top most important criteria for which there is medical consensus. Make them operatable with specific thresholds.'
task_expected_output = 'A brief summary of clinical criteria for T2D severity along with references, optimized for EHR.'

# Task 1: GPT-4o Search (Runs asynchronously)
task_gpt = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_gpt,
    async_execution=True # <--- Key for parallel execution
)

# Task 2: DeepSeek Search (Runs asynchronously)
task_deepseek = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_deepseek,
    async_execution=True # <--- Key for parallel execution
)

# Task 3: Claude Consolidation (Runs sequentially after the first two)
task_consolidate = Task(
    description='Review the research provided by the two clinical researchers. Compare their findings, identify the strongest consensus, and produce a final, unified framework for T2D severity phenotyping.',
    expected_output='A definitive, consolidated guide of operatable EHR criteria for T2D severity with specific thresholds and references.',
    agent=consolidator_claude,
    context=[task_gpt, task_deepseek] # <--- Feeds the output of both previous tasks to Claude
)

# ── 5. Run the Crew ───────────────────────────────────────────
print("🚀 Kickoff CrewAI Parallel Execution...")

crew = Crew(
    agents=[researcher_gpt, researcher_deepseek, consolidator_claude],
    tasks=[task_gpt, task_deepseek, task_consolidate],
    verbose=True
)

import sys
import io

# ── Capture ALL console output ────────────────────────────────
log_capture = io.StringIO()

class TeeOutput:
    """Writes to both console AND a string buffer."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
    def flush(self):
        for s in self.streams:
            s.flush()

# Start capturing
original_stdout = sys.stdout
sys.stdout = TeeOutput(original_stdout, log_capture)

try:
    result = crew.kickoff()

    # Stop capturing
    sys.stdout = original_stdout
    full_log = log_capture.getvalue()

    print("\n✅ CrewAI Execution Complete!")
    print(f"📊 Full log captured: {len(full_log)} chars")
    print("\n================ FINAL CONSOLIDATED OUTPUT ================\n")
    print(result)

    # ── Save everything ───────────────────────────────────────
    from google.colab import files
    import shutil

    # 1. Save the FULL verbose log (everything on screen)
    with open(f"{OUTPUT_DIR}/phase1/full_execution_log.txt", "w") as f:
        f.write(full_log)

    # 2. Save consolidated framework
    with open(f"{OUTPUT_DIR}/phase1/consolidated_framework/consolidated_framework.txt", "w") as f:
        f.write(str(result))

    # 3. Save individual agent final outputs
    for task_obj, name in [(task_gpt, "openai"), (task_deepseek, "deepseek"), (task_consolidate, "consolidated_framework")]:
        try:
            with open(f"{OUTPUT_DIR}/phase1/{name}/final_output.txt", "w") as f:
                f.write(str(task_obj.output))
            print(f"✅ Saved {name}: {len(str(task_obj.output))} chars")
        except Exception as e:
            print(f"⚠️ Could not save {name}: {e}")

    # 4. Zip and download
    shutil.make_archive("/content/phase1_outputs", "zip", f"{OUTPUT_DIR}/phase1")
    files.download("/content/phase1_outputs.zip")
    print("📥 Downloading!")

except Exception as e:
    sys.stdout = original_stdout
    print(f"❌ CrewAI failed: {e}")
    import traceback
    traceback.print_exc()